In [1]:
import numpy as np
import joblib
import tensorflow as tf

# 1. LOAD THE SAVED MODELS
# Load the scaler and Random Forest
scaler = joblib.load('finsaathi_scaler.pkl')
rf_model = joblib.load('finsaathi_rf_model.pkl')

# Load the Neural Network using the Interpreter
# (Ignoring the deprecation warning for now as it still works in TF 2.x)
interpreter = tf.lite.Interpreter(model_path="finsaathi_nn.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# 2. THE PREDICTION FUNCTION
def get_finsaathi_advice(age, income, dependents, occupation_id, city_tier_id, rent):
    # Prepare input data
    input_data = np.array([[age, income, dependents, occupation_id, city_tier_id, rent]])
    
    # Scale the data using the saved scaler
    scaled_data = scaler.transform(input_data).astype(np.float32)

    # --- MODEL A: RANDOM FOREST ---
    rf_prob = rf_model.predict_proba(scaled_data)[:, 1][0]

    # --- MODEL B: NEURAL NETWORK ---
    interpreter.set_tensor(input_details[0]['index'], scaled_data)
    interpreter.invoke()
    nn_prob = interpreter.get_tensor(output_details[0]['index'])[0][0]

    # --- ENSEMBLE RESULT ---
    final_score = (rf_prob + nn_prob) / 2
    return final_score

# 3. TEST RUN
# Input format: Age, Income, Dependents, OccupationID, CityTierID, Rent
score = get_finsaathi_advice(25, 30000, 2, 3, 1, 5000)

if score > 0.5:
    print(f"Confidence: {score:.2f} - Rohit is on track with savings!")
else:
    print(f"Confidence: {score:.2f} - Alert: Potential Budget Leak detected.")

c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle

Confidence: 0.98 - Rohit is on track with savings!


c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [6]:
pip install --upgrade scikit-learn joblib ai-edge-litert


   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
    --------------------------------------- 0.3/12.8 MB ? eta -:--:--
    --------------------------------------- 0.3/12.8 MB ? eta -:--:--
    --------------------------------------- 0.3/12.8 MB ? eta -:--:--
    --------------------------------------- 0.3/12.8 MB ? eta -:--:--
    --------------------------------------- 0.3/12.8 MB ? eta -:--:--
    ---------------

In [7]:
import numpy as np
import joblib
import warnings

# -------------------------------
# 1. SAFE IMPORT FOR EDGE AI (2026+)
# -------------------------------
try:
    import ai_edge_litert.interpreter as litert
except ImportError:
    import tensorflow.lite as litert
    warnings.warn("Using deprecated tf.lite.Interpreter. Install ai-edge-litert for future compatibility.")

# -------------------------------
# 2. LOAD MODELS SAFELY
# -------------------------------
def load_models():
    try:
        scaler = joblib.load('finsaathi_scaler.pkl')
        rf_model = joblib.load('finsaathi_rf_model.pkl')
        return scaler, rf_model
    except Exception as e:
        raise RuntimeError(f"❌ Model loading failed: {e}")

scaler, rf_model = load_models()

# -------------------------------
# 3. INITIALIZE INTERPRETER
# -------------------------------
try:
    interpreter = litert.Interpreter(model_path="finsaathi_nn.tflite")
    interpreter.allocate_tensors()
except Exception as e:
    raise RuntimeError(f"❌ TFLite model failed to load: {e}")

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# -------------------------------
# 4. MAIN PREDICTION FUNCTION
# -------------------------------
def get_finsaathi_advice(age, income, dependents, occupation_id, city_tier_id, rent):
    
    # ✅ Input validation
    input_data = np.array([[age, income, dependents, occupation_id, city_tier_id, rent]])
    
    if input_data.shape != (1, 6):
        raise ValueError("❌ Input must contain exactly 6 features")
    
    # ✅ Scale input
    try:
        scaled_data = scaler.transform(input_data).astype(np.float32)
    except Exception as e:
        raise RuntimeError(f"❌ Scaling failed: {e}")

    # -------------------------------
    # MODEL A: RANDOM FOREST
    # -------------------------------
    try:
        rf_prob = rf_model.predict_proba(scaled_data)[:, 1][0]
    except Exception as e:
        raise RuntimeError(f"❌ RF prediction failed: {e}")

    # -------------------------------
    # MODEL B: NEURAL NETWORK (LiteRT)
    # -------------------------------
    try:
        interpreter.set_tensor(input_details[0]['index'], scaled_data)
        interpreter.invoke()

        nn_output = interpreter.get_tensor(output_details[0]['index'])
        nn_prob = float(nn_output.flatten()[0])  # ✅ FIXED SHAPE ISSUE

    except Exception as e:
        raise RuntimeError(f"❌ NN prediction failed: {e}")

    # -------------------------------
    # ENSEMBLE (Weighted for better accuracy)
    # -------------------------------
    final_score = (0.6 * rf_prob) + (0.4 * nn_prob)

    return final_score


# -------------------------------
# 5. USER-FRIENDLY OUTPUT
# -------------------------------
def get_message(score, name="User"):
    if score > 0.75:
        return f"✅ Confidence: {score:.2f} - {name} is in excellent financial shape!"
    elif score > 0.5:
        return f"⚠️ Confidence: {score:.2f} - {name} is stable but can improve savings."
    else:
        return f"🚨 Confidence: {score:.2f} - Potential budget leak detected."


# -------------------------------
# 6. TEST RUN
# -------------------------------
if __name__ == "__main__":
    try:
        score = get_finsaathi_advice(25, 30000, 2, 3, 1, 5000)
        print(get_message(score, name="Rohit"))
    except Exception as e:
        print(e)

✅ Confidence: 0.98 - Rohit is in excellent financial shape!


c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle

In [9]:
import joblib

# Load old models
scaler = joblib.load('finsaathi_scaler.pkl')
rf_model = joblib.load('finsaathi_rf_model.pkl')

# Save new version-compatible models
joblib.dump(scaler, 'finsaathi_scaler_v2.pkl')
joblib.dump(rf_model, 'finsaathi_rf_model_v2.pkl')

print("✅ New v2 models saved!")

c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle

✅ New v2 models saved!


In [10]:
import numpy as np
import joblib
import warnings

# -------------------------------
# 1. EDGE AI IMPORT (FUTURE SAFE)
# -------------------------------
try:
    import ai_edge_litert.interpreter as litert
except ImportError:
    import tensorflow.lite as litert
    warnings.warn("⚠️ Using deprecated tf.lite.Interpreter. Install ai-edge-litert.")

# -------------------------------
# 2. LOAD MODELS SAFELY
# -------------------------------
def load_models():
    try:
        scaler = joblib.load('finsaathi_scaler_v2.pkl')  # ✅ use re-saved models
        rf_model = joblib.load('finsaathi_rf_model_v2.pkl')
        return scaler, rf_model
    except Exception as e:
        raise RuntimeError(f"❌ Model loading failed: {e}")

scaler, rf_model = load_models()

# -------------------------------
# 3. LOAD TFLITE MODEL
# -------------------------------
try:
    interpreter = litert.Interpreter(model_path="finsaathi_nn.tflite")
    interpreter.allocate_tensors()
except Exception as e:
    raise RuntimeError(f"❌ TFLite model error: {e}")

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# -------------------------------
# 4. MAIN PREDICTION FUNCTION
# -------------------------------
def get_finsaathi_advice(age, income, dependents, occupation_id, city_tier_id, rent):
    
    # Input validation
    input_data = np.array([[age, income, dependents, occupation_id, city_tier_id, rent]])
    
    if input_data.shape != (1, 6):
        raise ValueError("❌ Input must have 6 features")

    # Scaling
    scaled_data = scaler.transform(input_data).astype(np.float32)

    # --- RANDOM FOREST ---
    rf_prob = rf_model.predict_proba(scaled_data)[:, 1][0]

    # --- NEURAL NETWORK ---
    interpreter.set_tensor(input_details[0]['index'], scaled_data)
    interpreter.invoke()

    nn_output = interpreter.get_tensor(output_details[0]['index'])
    nn_prob = float(nn_output.flatten()[0])  # ✅ safe extraction

    # --- ENSEMBLE (weighted) ---
    final_score = (0.7 * rf_prob) + (0.3 * nn_prob)

    # --- CONTROL OVERCONFIDENCE ---
    final_score = max(0.05, min(0.95, final_score))

    return final_score


# -------------------------------
# 5. HUMAN-FRIENDLY MESSAGE
# -------------------------------
def get_message(score, name="User"):
    if score > 0.75:
        return f"✅ Confidence: {score:.2f} - {name} is in excellent financial shape!"
    elif score > 0.5:
        return f"⚠️ Confidence: {score:.2f} - {name} is stable but can improve savings."
    else:
        return f"🚨 Confidence: {score:.2f} - Potential budget leak detected."


# -------------------------------
# 6. SMART FINANCIAL ADVICE
# -------------------------------
def generate_advice(age, income, dependents, rent, score):
    advice = []

    if rent > 0.4 * income:
        advice.append("🏠 Rent is too high (>40% of income). Consider reducing it.")

    if income < 20000:
        advice.append("💰 Income is low. Try to increase income sources.")

    if dependents >= 3:
        advice.append("👨‍👩‍👧 High dependents. Maintain insurance & emergency fund.")

    if score < 0.5:
        advice.append("🚨 High financial risk. Cut unnecessary expenses.")
    elif score < 0.75:
        advice.append("⚠️ Moderate stability. Improve savings discipline.")
    else:
        advice.append("✅ Good financial health. Consider investments (SIP, mutual funds).")

    return advice


# -------------------------------
# 7. EXPLAIN MODEL (FEATURE IMPORTANCE)
# -------------------------------
def explain_model():
    try:
        features = ["Age", "Income", "Dependents", "Occupation", "City Tier", "Rent"]
        importance = rf_model.feature_importances_

        print("\n🔍 Feature Importance:")
        for f, imp in zip(features, importance):
            print(f"{f}: {imp:.2f}")
    except:
        print("⚠️ Feature importance not available.")


# -------------------------------
# 8. TEST MULTIPLE SCENARIOS
# -------------------------------
def test_model():
    test_cases = [
        (22, 15000, 0, 1, 1, 2000),
        (30, 50000, 2, 3, 2, 15000),
        (40, 80000, 3, 4, 1, 30000),
        (28, 25000, 1, 2, 3, 10000),
        (35, 100000, 4, 5, 1, 50000)
    ]

    print("\n📊 Model Testing:")
    for case in test_cases:
        score = get_finsaathi_advice(*case)
        print(case, "→", round(score, 2))


# -------------------------------
# 9. MAIN RUN
# -------------------------------
if __name__ == "__main__":
    try:
        # Example user
        age, income, dependents, occupation, city, rent = 25, 30000, 2, 3, 1, 5000

        score = get_finsaathi_advice(age, income, dependents, occupation, city, rent)

        print(get_message(score, "Rohit"))

        print("\n📊 Financial Advice:")
        advice = generate_advice(age, income, dependents, rent, score)
        for tip in advice:
            print("-", tip)

        explain_model()
        test_model()

    except Exception as e:
        print(e)

✅ Confidence: 0.95 - Rohit is in excellent financial shape!

📊 Financial Advice:
- ✅ Good financial health. Consider investments (SIP, mutual funds).

🔍 Feature Importance:
Age: 0.18
Income: 0.30
Dependents: 0.07
Occupation: 0.05
City Tier: 0.09
Rent: 0.31

📊 Model Testing:
(22, 15000, 0, 1, 1, 2000) → 0.95
(30, 50000, 2, 3, 2, 15000) → 0.82
(40, 80000, 3, 4, 1, 30000) → 0.6
(28, 25000, 1, 2, 3, 10000) → 0.86
(35, 100000, 4, 5, 1, 50000) → 0.52
